# Experiment: Estimating Bitcoin VaR and CVaR with Machine Learning
- There are several approaches to estimating VaR and CVaR with machine learning.
- In this experiment, the focus is on achieving high precision using neural network techniques.
- First, we use a distinct LSTM (Long Short-Term Memory) model for each time horizon (1 quarter, 1 year, and 5 years).
- These models predict the mean and standard deviation of future log returns based on the historical log returns.
- Then, we perform three separate Monte Carlo simulations using the respective mean and standard deviation.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, LSTM, Input
from tensorflow.keras.optimizers import Adam

In [ ]:
# Set charts theme
sns.set_theme(style="darkgrid", rc={"grid.alpha": 1/3})
plt.style.use("dark_background")

# Save chart as png function
def save_chart_as_png(filename: str) -> None:
    plt.savefig(
        f"../images/{filename}.png",
        format="png",
        dpi=300,
        orientation="landscape",
        bbox_inches="tight",
    )

In [ ]:
# Get bitcoin df with date as index
df_btc = pd.read_csv("../data/BTC.csv", index_col="date", parse_dates=True)

# Get log price change (log returns)
df_btc["price_change_log"] = np.log(df_btc["price"] / df_btc["price"].shift(1))

# Get log returns as simple array
log_returns = df_btc["price_change_log"].dropna().values

## Predict the mean and std using neural networks 🤖

### Data preparation functions

In [ ]:
def get_data_arrays(input_window: int, predict_window: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Prepare data arrays for the model.
    input_window: Number of past time steps to use as input for predictions.
    predict_window: Number of future time steps for which mean and std are predicted.
    """
    
    # Ensure there is enough data for the specified windows
    if len(log_returns) <= input_window + predict_window:
        raise ValueError("Not enough data for the specified input and prediction windows.")


    # Doing this in an easy to undersand way (without np.lib.stride_tricks...)
    X, y_mean, y_std = [], [], []
    for start_idx in range(len(log_returns) - input_window - predict_window):
        # Define the indices for input and output sequences
        end_idx_x = start_idx + input_window
        end_idx_y = end_idx_x + predict_window

        # Extract input sequence
        X.append(log_returns[start_idx:end_idx_x])

        # Extract output sequence and calc mean and std
        future_values = log_returns[end_idx_x:end_idx_y]
        y_mean.append(future_values.mean())
        y_std.append(future_values.std())

    # Convert lists to np arrays
    X = np.array(X)[..., np.newaxis]  # Add feature dimension for LSTM input
    y_mean = np.array(y_mean)
    y_std = np.array(y_std)

    print(f"X shape: {X.shape}, y_mean shape: {y_mean.shape}, y_std shape: {y_std.shape}")

    return X, y_mean, y_std

In [ ]:
def split_data(data: tuple[np.ndarray,], train_ratio: float = 0.8, val_ratio: float = 0.1) -> tuple[tuple[np.ndarray,],]:
    """
    Split the data into training, validation, and testing sets chronologically.
    data: Tuple of arrays (X, y_mean, y_std).
    train_ratio: Proportion of data to use for training.
    val_ratio: Proportion of data to use for validation.
    """
    
    # Validate ratios
    if train_ratio + val_ratio >= 1.0:
        raise ValueError("The sum of train_ratio and val_ratio must be less than 1.")

    # Extract data from tuple
    X, y_mean, y_std = data

    # Calc split indices
    first_split_index = int(len(X) * train_ratio)
    second_split_index = int(len(X) * (train_ratio + val_ratio))

    # Split the data
    X_train, y_mean_train, y_std_train = X[:first_split_index], y_mean[:first_split_index], y_std[:first_split_index]
    X_val, y_mean_val, y_std_val = X[first_split_index:second_split_index], y_mean[first_split_index:second_split_index], y_std[first_split_index:second_split_index]
    X_test, y_mean_test, y_std_test = X[second_split_index:], y_mean[second_split_index:], y_std[second_split_index:]
    
    # Print shapes for debugging
    def print_shapes(prefix, X, y_mean, y_std):
        print(f"Raw {prefix} -> X: {X.shape}, y_mean: {y_mean.shape}, y_std: {y_std.shape}")

    print_shapes("Train", X_train, y_mean_train, y_std_train)
    print_shapes("Val", X_val, y_mean_val, y_std_val)
    print_shapes("Test", X_test, y_mean_test, y_std_test)

    return (
        (X_train, y_mean_train, y_std_train),
        (X_val, y_mean_val, y_std_val),
        (X_test, y_mean_test, y_std_test),
    )

In [ ]:
def normalize_data(
    train_data_raw: tuple[np.ndarray,], val_data_raw: tuple[np.ndarray,], test_data_raw: tuple[np.ndarray,]
) -> tuple[tuple[np.ndarray,],]:
    """
    Normalize the training, validation, and testing data (after the split to avoid data leakage).
    """
    
    # Extract data from tuples
    X_train_raw, y_mean_train_raw, y_std_train_raw = train_data_raw
    X_val_raw, y_mean_val_raw, y_std_val_raw = val_data_raw
    X_test_raw, y_mean_test_raw, y_std_test_raw = test_data_raw

    # Calc mean and std of the training data only
    X_mean, X_std = X_train_raw.mean(axis=(0, 1)), X_train_raw.std(axis=(0, 1))
    y_mean_mean, y_mean_std = y_mean_train_raw.mean(), y_mean_train_raw.std()
    y_std_mean, y_std_std = y_std_train_raw.mean(), y_std_train_raw.std()

    def normalize(array, mean, std):
        return (array - mean) / std

    # Normalize X
    X_train = normalize(X_train_raw, X_mean, X_std)
    X_val = normalize(X_val_raw, X_mean, X_std)
    X_test = normalize(X_test_raw, X_mean, X_std)

    # Normalize y_mean
    y_mean_train = normalize(y_mean_train_raw, y_mean_mean, y_mean_std)
    y_mean_val = normalize(y_mean_val_raw, y_mean_mean, y_mean_std)
    y_mean_test = normalize(y_mean_test_raw, y_mean_mean, y_mean_std)

    # Normalize y_std
    y_std_train = normalize(y_std_train_raw, y_std_mean, y_std_std)
    y_std_val = normalize(y_std_val_raw, y_std_mean, y_std_std)
    y_std_test = normalize(y_std_test_raw, y_std_mean, y_std_std)

    def print_shapes(prefix, X, y_mean, y_std):
        print(f"Normalized {prefix} -> X: {X.shape}, y_mean: {y_mean.shape}, y_std: {y_std.shape}")

    print_shapes("Train", X_train, y_mean_train, y_std_train)
    print_shapes("Val", X_val, y_mean_val, y_std_val)
    print_shapes("Test", X_test, y_mean_test, y_std_test)

    return (
        (X_train, y_mean_train, y_std_train),
        (X_val, y_mean_val, y_std_val),
        (X_test, y_mean_test, y_std_test),
    )

### Model functions

In [ ]:
def plot_training_history(history) -> None:
    plt.figure(figsize=(10, 6))

    sns.lineplot(x=range(len(history.history["loss"])), y=history.history["loss"], label="Training Loss", color="#00f8ff")
    sns.lineplot(x=range(len(history.history["val_loss"])), y=history.history["val_loss"], label="Validation Loss", color="#ff5b00")

    plt.title("Training and Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")

    plt.legend()

In [ ]:
def evaluate_model(test_data: tuple[np.ndarray,], train_data_raw: tuple[np.ndarray,], model) -> None:
    """
    Evaluate the model on test data. And compare actual and predicted mean and std from a test data point.
    """
    
    # Extract data from tuples
    X_test, y_mean_test, y_std_test = test_data
    X_train_raw, y_mean_train_raw, y_std_train_raw = train_data_raw
    
    # Evaluate the model on test data
    total_loss, mean_loss, std_loss = model.evaluate(
        X_test,
        {"mean_output": y_mean_test, "std_output": y_std_test},
        verbose=0,
    )

    # Calc RMSE for the losses
    total_rmse = np.sqrt(total_loss)
    mean_rmse = np.sqrt(mean_loss)
    std_rmse = np.sqrt(std_loss)

    print(f"\nTotal Loss: {total_loss:.4f}, Mean Loss: {mean_loss:.4f}, Std Loss: {std_loss:.4f}")
    print(f"Total RMSE: {total_rmse:.4f}, Mean RMSE: {mean_rmse:.4f}, Std RMSE: {std_rmse:.4f}")

    # Get last sample of test data
    X_test_last = np.expand_dims(X_test[-1], axis=0)
    y_mean_test_last = y_mean_test[-1]
    y_std_test_last = y_std_test[-1]
    
    # Calc mean and std of the training data
    X_mean, X_std = X_train_raw.mean(axis=(0, 1)), X_train_raw.std(axis=(0, 1))
    y_mean_mean, y_mean_std = y_mean_train_raw.mean(), y_mean_train_raw.std()
    y_std_mean, y_std_std = y_std_train_raw.mean(), y_std_train_raw.std()

    # Predict normalized mean and std from last sample of testing set
    predicted_mean_normalized, predicted_std_normalized = model.predict(X_test_last, verbose=0)

    # Denormalize predictions
    predicted_mean = predicted_mean_normalized.item() * y_mean_std + y_mean_mean
    predicted_std = predicted_std_normalized.item() * y_std_std + y_std_mean

    # Denormalize actual values from last sample
    mean = y_mean_test_last * y_mean_std + y_mean_mean
    std = y_std_test_last * y_std_std + y_std_mean

    print("\nComparison of last sample:")
    print(f"Predicted Mean: {predicted_mean:.5f}, Actual Mean: {mean:.5f}")    
    print(f"Predicted Std: {predicted_std:.5f}, Actual Std: {std:.5f}")    

In [ ]:
def predict_future(input_window: int, train_data_raw: tuple[np.ndarray,], model) -> tuple[float, float]:
    """
    Predict the mean and standard deviation of future values based on the last input_window of log returns.
    """
    
    # Extract the last input_window of log returns
    last_returns = log_returns[-input_window:]

    # Ensure correct shape for model input
    last_returns_raw = last_returns.reshape((1, input_window, 1))
    
    # Extract data from tuple
    X_train_raw, y_mean_train_raw, y_std_train_raw = train_data_raw
    
    # Calc mean and std of the training data
    X_mean, X_std = X_train_raw.mean(axis=(0, 1)), X_train_raw.std(axis=(0, 1))
    y_mean_mean, y_mean_std = y_mean_train_raw.mean(), y_mean_train_raw.std()
    y_std_mean, y_std_std = y_std_train_raw.mean(), y_std_train_raw.std()

    # Normalize the returns
    last_returns_normalized = (last_returns_raw - X_mean) / X_std

    # Predict normalized mean and std
    predicted_mean_normalized, predicted_std_normalized = model.predict(last_returns_normalized, verbose=0)
    
    # Denormalize predictions
    predicted_mean = predicted_mean_normalized.item() * y_mean_std + y_mean_mean
    predicted_std = predicted_std_normalized.item() * y_std_std + y_std_mean

    # Print the predictions
    print(f"Predicted Mean: {predicted_mean:.5f}, Predicted Std: {predicted_std:.5f}")
    
    return predicted_mean, predicted_std

### Predict the mean and standard deviation for the next 90 days

In [ ]:
input_window = 365 * 2  # best results
predict_window = 90

data_raw = get_data_arrays(input_window, predict_window)

In [ ]:
train_data_raw, val_data_raw, test_data_raw = split_data(data_raw)

In [ ]:
train_data, val_data, test_data = normalize_data(train_data_raw, val_data_raw, test_data_raw)

In [ ]:
X_train, y_mean_train, y_std_train = train_data
X_val, y_mean_val, y_std_val = val_data

*The model code could also be refactored, although it would make it more difficult to read.*

In [ ]:
# Model
# Input layer
input_layer = Input(shape=(X_train.shape[1], 1))

# First LSTM layer
x = LSTM(128, activation="tanh", return_sequences=True)(input_layer) # tanh gives the best results stabilizing
x = LayerNormalization()(x)  # improve stability and efficiency

# Second LSTM layer
x = LSTM(64, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)

# Third LSTM layer
x = LSTM(32, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)

# Fourth LSTM layer
x = LSTM(16, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)

# Fifth LSTM layer
x = LSTM(8, activation="tanh", return_sequences=False)(x)
x = LayerNormalization()(x)

# Output layers for mean and std
output_mean = Dense(1, activation="linear", name="mean_output")(x)
output_std = Dense(1, activation="softplus", name="std_output")(x)  # std must be positive

model = Model(inputs=input_layer, outputs=[output_mean, output_std])
model.compile(
    optimizer="adam",  # better results than rmsprop
    loss={"mean_output": "mse", "std_output": "mse"},
)

model.summary()

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1)  # reduce learning rate to 50% of last value after 5 epochs without better val_loss
early_stopping = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)  # stop 3 epochs after the reduced learning rate if not better val_loss

# Fit model
history = model.fit(
    X_train,
    {"mean_output": y_mean_train, "std_output": y_std_train},
    validation_data=(
        X_val,
        {"mean_output": y_mean_val, "std_output": y_std_val},
    ),
    epochs=50,
    batch_size=64,  # 64 gives the best balance
    shuffle=False,  # important for time series
    callbacks=[reduce_lr, early_stopping],
)

In [ ]:
plot_training_history(history)

In [ ]:
evaluate_model(test_data, train_data_raw, model)

In [ ]:
# Next quarter mean and std prediction
predicted_mean_90, predicted_std_90 = predict_future(input_window, train_data_raw, model)

### Predict the mean and standard deviation for the next 365 days

In [ ]:
input_window = 365 * 2  # best results
predict_window = 365

data_raw = get_data_arrays(input_window, predict_window)

In [ ]:
train_data_raw, val_data_raw, test_data_raw = split_data(data_raw)

In [ ]:
train_data, val_data, test_data = normalize_data(train_data_raw, val_data_raw, test_data_raw)

In [ ]:
X_train, y_mean_train, y_std_train = train_data
X_val, y_mean_val, y_std_val = val_data

In [ ]:
# Model
# Input layer
input_layer = Input(shape=(X_train.shape[1], 1))

# First LSTM layer
x = LSTM(64, activation="tanh", return_sequences=True)(input_layer) # tanh gives the best results stabilizing
x = LayerNormalization()(x)  # improve stability and efficiency
x = Dropout(0.4)(x)  # to prevent overfitting

# Second LSTM layer
x = LSTM(32, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)
x = Dropout(0.4)(x)

# Third LSTM layer
x = LSTM(16, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)
x = Dropout(0.4)(x)

# Fourth LSTM layer
x = LSTM(8, activation="tanh", return_sequences=True)(x)
x = LayerNormalization()(x)
x = Dropout(0.4)(x)

# Fifth LSTM layer
x = LSTM(4, activation="tanh", return_sequences=False)(x)
x = LayerNormalization()(x)
x = Dropout(0.4)(x)

# Output layers for mean and std
output_mean = Dense(1, activation="linear", name="mean_output")(x)
output_std = Dense(1, activation="softplus", name="std_output")(x)  # std must be positive

model = Model(inputs=input_layer, outputs=[output_mean, output_std])
model.compile(
    optimizer="adam",  # better results than rmsprop
    loss={"mean_output": "mse", "std_output": "mse"},
)

model.summary()

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1)  # reduce learning rate to 50% of last value after 5 epochs without better val_loss
early_stopping = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)  # stop 3 epochs after the reduced learning rate if not better val_loss

# Fit model
history = model.fit(
    X_train,
    {"mean_output": y_mean_train, "std_output": y_std_train},
    validation_data=(
        X_val,
        {"mean_output": y_mean_val, "std_output": y_std_val},
    ),
    epochs=50,
    batch_size=64,  # 64 gives the best balance
    shuffle=False,  # important for time series
    callbacks=[reduce_lr, early_stopping],
)

In [ ]:
plot_training_history(history)

In [ ]:
evaluate_model(test_data, train_data_raw, model)

In [ ]:
# Next year mean and std prediction
predicted_mean_365, predicted_std_365 = predict_future(input_window, train_data_raw, model)

### Predict the mean and standard deviation for the next 1825 days (5 years)

In [ ]:
input_window = 365 * 7  # best results
predict_window = 365 * 5

data_raw = get_data_arrays(input_window, predict_window)

In [ ]:
train_data_raw, val_data_raw, test_data_raw = split_data(data_raw)

In [ ]:
train_data, val_data, test_data = normalize_data(train_data_raw, val_data_raw, test_data_raw)

In [ ]:
X_train, y_mean_train, y_std_train = train_data
X_val, y_mean_val, y_std_val = val_data

In [ ]:
# Model
# Input layer
input_layer = Input(shape=(X_train.shape[1], 1))

# First LSTM layer
x = LSTM(128, activation="tanh", return_sequences=True)(input_layer) # tanh gives the best results stabilizing
x = LayerNormalization()(x)  # improve stability and efficiency
x = Dropout(0.8)(x)  # to prevent overfitting

# Second LSTM layer
x = LSTM(64, activation="tanh", return_sequences=False)(input_layer)
x = LayerNormalization()(x)
x = Dropout(0.8)(x) 

# Output layers for mean and std
output_mean = Dense(1, activation="linear", name="mean_output")(x)
output_std = Dense(1, activation="softplus", name="std_output")(x)  # std must be positive

model = Model(inputs=input_layer, outputs=[output_mean, output_std])
model.compile(
    optimizer=Adam(learning_rate=0.00025),  # reduced learning rate since there are much fewer samples
    loss={"mean_output": "mse", "std_output": "mse"},
)

model.summary()

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1)  # reduce learning rate to 50% of last value after 5 epochs without better val_loss
early_stopping = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)  # stop 3 epochs after the reduced learning rate if not better val_loss

# Fit model
history = model.fit(
    X_train,
    {"mean_output": y_mean_train, "std_output": y_std_train},
    validation_data=(
        X_val,
        {"mean_output": y_mean_val, "std_output": y_std_val},
    ),
    epochs=50,
    batch_size=32,  # 32 is better here
    shuffle=False,  # important for time series
    callbacks=[reduce_lr, early_stopping],
)

In [ ]:
plot_training_history(history)

In [ ]:
evaluate_model(test_data, train_data_raw, model)

In [ ]:
# Next 5 years mean and std prediction
predicted_mean_1825, predicted_std_1825 = predict_future(input_window, train_data_raw, model)

### Compare the mean and standard deviation predictions

In [ ]:
time_horizons = (90, predicted_mean_90, predicted_std_90), (365, predicted_mean_365, predicted_std_365), (1825, predicted_mean_1825, predicted_std_1825)
table = []

for days, mean, std in time_horizons:
    table.append({
        "Time Horizon": f"{days} days",
        "Predicted Mean": mean,
        "Predicted Std": std,
    })

pd.DataFrame(table)

## Use predicted mean and std to get VaR and CVaR using Monte Carlo method 🚨

In [ ]:
# Calculate VaR and CVaR based for specific aggregated returns and confidence interval
def calculate_var_and_cvar(aggregated_returns: pd.Series, confidence_interval: float) -> tuple[float, float]:
    # Convert confidence interval to the corresponding percentile for VaR calculation
    percentile = (1 - confidence_interval) * 100
    
    # Calculate the historical VaR as the negative value at the specified percentile of aggregated returns
    var = -np.percentile(aggregated_returns, percentile).round(3)
    
    # Calculate CVaR as the negative mean of returns that are less than or equal to the calculated VaR
    cvar = -aggregated_returns[aggregated_returns <= -var].mean().round(3)
    
    return var, cvar


# Get table with VaR and CVaR for a specific confidence interval for different time horizons using the Monte Carlo method
num_simulations = 25_000
aggregated_returns_list = []
var_cvar_results = []

for days, mean, std in time_horizons:
    # Simulate future returns using a normal distribution (output is array of x days by y simulations)
    simulated_returns = np.random.normal(mean, std, (num_simulations, days))
    
    # Aggregate returns over the time horizon (sum x days of each simulation)
    aggregated_returns = simulated_returns.sum(axis=1)

    # Append aggregated returns to use in histograms
    aggregated_returns_list.append(aggregated_returns)
    
    # Calculate VaR and CVaR using the aggregated returns for 95% and 99% confidence interval
    var_95, cvar_95 = calculate_var_and_cvar(aggregated_returns, 0.95)
    var_99, cvar_99 = calculate_var_and_cvar(aggregated_returns, 0.99)
    
    # Append results to the list
    var_cvar_results.append({
        "Time Horizon": f"{days} days",
        "VaR (95%)": var_95,
        "CVaR (95%)": cvar_95,
        "VaR (99%)": var_99,
        "CVaR (99%)": cvar_99,
    })

pd.DataFrame(var_cvar_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

colors = ["#40b0e0", "#40e07f", "#b1e040"]
titles = [f"1-quarter Simulated Returns", f"1-year Simulated Returns", f"5-year Simulated Returns"]

for i, aggregated_returns in enumerate(aggregated_returns_list):
    sns.histplot(aggregated_returns, stat="probability", binwidth=0.1, binrange=(-1, 1), color=colors[i], edgecolor="white", alpha=0.75, ax=axes[i])
    
    axes[i].axvline(np.percentile(aggregated_returns, 5), color="orange", linewidth=1.5, linestyle="--", label="VaR at 95% Confidence Level")
    axes[i].axvline(np.percentile(aggregated_returns, 1), color="red", linewidth=1.5, linestyle="--", label="VaR at 99% Confidence Level")

    axes[i].set_xlim(-1, 1)
    axes[i].tick_params(axis="both", labelsize=10) 
    
    axes[i].set_title(f"Distribution of the {titles[i]}")
    axes[i].set_xlabel(None)
    
axes[0].set_ylabel("Probability")

axes[1].legend(loc="upper right", fontsize=10)

plt.tight_layout()

save_chart_as_png("3.3_BTC_ml_var")

**Key takeaways:**